# Gold Set Sampling — XSAMSum (zh) BART Baseline

Filter precomputed baseline scores to a 50-sample gold set, then extract the corresponding dialogues from the original `test.json`:

1. Load `pair_scores_zh_XSAMSum_bart.csv` (already has `rougeL` and `bs_f1_raw`)
2. Drop samples in the bottom 5% or top 5% on **either** ROUGE-L or BERTScore-F1
3. Randomly sample 50 from the remaining ~90% with `random_state=42`
4. Look up the matching entries in `test.json` and emit them with `summary_de` removed

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42

INPUT_PATH      = "pair_scores_zh_XSAMSum_bart.csv"
TEST_JSON_PATH  = "test.json"                          # original XSAMSum test split
OUTPUT_CSV      = "gold_set_50_zh_XSAMSum_bart.csv"
OUTPUT_JSON     = "gold_set_50_zh_XSAMSum_bart.json"   # dialogues + retained summary fields

LOWER_PCT = 5
UPPER_PCT = 95
N_GOLD = 50

## 1. Load scores

In [2]:
df = pd.read_csv(INPUT_PATH, index_col=0)
print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loaded 819 rows
Columns: ['reference', 'prediction', 'rouge1', 'rouge2', 'rougeL', 'bs_f1_raw']


,reference,prediction,rouge1,rouge2,rougeL,bs_f1_raw
0,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。,汉娜不太了解他 阿曼达建议汉娜发短信给他,28.57,7.69,28.57,67.52
1,埃里克和罗伯要在youtube上看一场单口相声。,Eric和Rob正在看一个俄国喜剧演员的脱口秀,21.05,0.00,21.05,63.59
2,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。,Lenny会买两条黑裤子和一条紫色的,15.79,0.00,15.79,58.65
3,艾玛很快就会回家，而且她会告诉威尔。,Emma今晚不想做晚饭 她马上回家 Will会来接她,24.00,8.70,24.00,63.13
4,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...,简从摩洛哥回来了 奥利和简周五下午6点会合吃午餐,32.26,10.00,22.58,68.02


In [3]:
df[["rougeL", "bs_f1_raw"]].describe()

,rougeL,bs_f1_raw
count,819.000000,819.000000
mean,30.714799,71.139841
std,15.356160,7.871887
min,0.000000,44.860000
25%,20.645000,65.755000
50%,28.570000,71.150000
75%,38.890000,75.985000
max,100.000000,97.520000


## 2. Compute P5 / P95 cutoffs and filter

A sample is kept only if **both** `rougeL` and `bs_f1_raw` fall within [P5, P95].

In [4]:
rouge_lo, rouge_hi = np.percentile(df["rougeL"],    [LOWER_PCT, UPPER_PCT])
bert_lo,  bert_hi  = np.percentile(df["bs_f1_raw"], [LOWER_PCT, UPPER_PCT])

print(f"ROUGE-L      [P{LOWER_PCT}, P{UPPER_PCT}] = [{rouge_lo:.2f}, {rouge_hi:.2f}]")
print(f"BERTScore-F1 [P{LOWER_PCT}, P{UPPER_PCT}] = [{bert_lo:.2f}, {bert_hi:.2f}]")

mask = (
    (df["rougeL"]    >= rouge_lo) & (df["rougeL"]    <= rouge_hi) &
    (df["bs_f1_raw"] >= bert_lo)  & (df["bs_f1_raw"] <= bert_hi)
)

filtered = df[mask].copy()
print(f"\nKept {len(filtered)} / {len(df)} samples ({len(filtered)/len(df):.1%})")

ROUGE-L      [P5, P95] = [9.30, 57.14]
BERTScore-F1 [P5, P95] = [58.50, 84.17]

Kept 699 / 819 samples (85.3%)


## 3. Random sample of 50 with fixed seed

In [5]:
assert len(filtered) >= N_GOLD, f"Only {len(filtered)} samples after filtering"

gold = filtered.sample(n=N_GOLD, random_state=SEED).sort_index()
print(f"Sampled {len(gold)} examples (seed={SEED})")
gold[["rougeL", "bs_f1_raw"]].describe()

Sampled 50 examples (seed=42)


,rougeL,bs_f1_raw
count,50.000000,50.000000
mean,26.438000,69.410800
std,10.543327,6.486415
min,10.810000,58.630000
25%,17.587500,64.010000
50%,25.580000,70.050000
75%,33.330000,75.332500
max,52.630000,80.210000


## 4. Sanity check — distribution shift

In [6]:
pd.DataFrame({
    "full":     [df["rougeL"].mean(),       df["bs_f1_raw"].mean()],
    "filtered": [filtered["rougeL"].mean(), filtered["bs_f1_raw"].mean()],
    "gold50":   [gold["rougeL"].mean(),     gold["bs_f1_raw"].mean()],
}, index=["ROUGE-L mean", "BERTScore-F1 mean"]).round(2)

,full,filtered,gold50
ROUGE-L mean,30.71,30.05,26.44
BERTScore-F1 mean,71.14,71.01,69.41


## 5. Extract corresponding dialogues from `test.json`

The CSV's row index aligns with the position of each example in `test.json`. We:

1. Load `test.json`
2. Verify alignment by checking that `test_data[i]["summary_zh"]` matches the CSV's `reference` for a few sampled rows
3. Pull the 50 selected entries and drop `summary_de` from each

In [8]:
with open(TEST_JSON_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data)} entries from {TEST_JSON_PATH}")
print(f"Fields in first entry: {list(test_data[0].keys())}")

Loaded 819 entries from test.json
Fields in first entry: ['dialogue', 'summary', 'summary_de', 'summary_zh']


In [9]:
# Verify CSV row index aligns with test.json position
# (spot-check on the gold sample — fast and sufficient)
mismatches = []
for i in gold.index:
    if test_data[i].get("summary_zh") != gold.loc[i, "reference"]:
        mismatches.append(i)

if mismatches:
    print(f"WARNING: {len(mismatches)} alignment mismatches at indices {mismatches[:5]}...")
    print("Index alignment between CSV and test.json may be off — investigate before proceeding.")
else:
    print(f"All {len(gold)} gold-set rows align with test.json by index ✓")

All 50 gold-set rows align with test.json by index ✓


In [10]:
# Build the gold dialogues, dropping summary_de
DROP_FIELDS = {"summary_de"}

gold_dialogues = []
for i in gold.index:
    entry = {k: v for k, v in test_data[i].items() if k not in DROP_FIELDS}
    # Track the original test.json index for traceability
    entry = {"test_index": int(i), **entry}
    gold_dialogues.append(entry)

print(f"Built {len(gold_dialogues)} gold-set entries")
print(f"Fields per entry: {list(gold_dialogues[0].keys())}")
gold_dialogues[0]

Built 50 gold-set entries
Fields per entry: ['test_index', 'dialogue', 'summary', 'summary_zh']


{'test_index': 59,
 'dialogue': "Laura: Where are you?\r\nPaul: Almost there.\r\nLaura: Which is?\r\nPaul: Close to the Mac.\r\nLaura: That's so far away!\r\nPaul: 15 mins\r\nLaura: I am not waiting any more, see you some other time.\r\nPaul: Please, wait!\r\nLaura: I've waited 30 minutes, 15 minutes ago you wrote you were almost here. This is too much.\r\nPaul: I am so sorry.\r\nLaura: I am not. ",
 'summary': 'Paul is late for a meeting with Laura and she refuses to wait any longer.',
 'summary_zh': '保罗和劳拉见面时迟到了，现在劳拉不想再等了。'}

## 6. Save

In [11]:
# Per-sample scores (CSV)
gold.to_csv(OUTPUT_CSV)
print(f"Wrote scores for {len(gold)} rows to {OUTPUT_CSV}")

# Dialogues with retained summary fields (JSON)
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(gold_dialogues, f, ensure_ascii=False, indent=2)
print(f"Wrote {len(gold_dialogues)} dialogues to {OUTPUT_JSON}")

Wrote scores for 50 rows to gold_set_50_zh_XSAMSum_bart.csv
Wrote 50 dialogues to gold_set_50_zh_XSAMSum_bart.json
